# 01 - Collect Remote Jobs (all fields)

## Objective
Collect remote job postings from **all fields** (IT, marketing, sales, design, finance...), so we can answer:
which fields are the most requested, how the seniority and salaries differ, and (as a deep dive) which IT skills are needed.

## Sources
| Source | What we ask for |
|---|---|
| RemoteOK API | latest ~100 jobs (all types) |
| Remotive API | **every category** (the notebook reads the official category list) |

## Rules (from the sources' terms)
- **Remotive:** mention Remotive as a source and link back to the job URL. Do not request too often: more than 2 requests per minute is blocked, and **one run per day is enough**. The code waits between requests, so a full run takes several minutes.
- **RemoteOK:** mention RemoteOK as a source and link back.
- Do **not** upload `data/raw/` to GitHub (add it to `.gitignore`). Publish only results, charts and the dashboard.

## How to use
Run the whole notebook. Run it again on other days to add new jobs. Each run **adds** to `jobs_raw.csv` and never deletes old jobs.

In [ ]:
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests

RAW_PATH = Path("../data/raw/jobs_raw.csv")
PAUSE_SECONDS = 35            # wait between Remotive requests (limit: 2 per minute)
# TARGET_JOBS = 1000            # goal for the project
# LIMIT_PER_CATEGORY = 200      # max jobs requested per Remotive category

HEADERS = {"User-Agent": "Mozilla/5.0 (personal data-analysis portfolio project)"}

COLUMNS = ["id", "date", "company", "position", "tags", "description", "location",
           "salary_min", "salary_max", "apply_url", "url",
           "source", "category", "collected_at"]

# used only if the category list cannot be downloaded (edit if a name is wrong)
FALLBACK_CATEGORIES = ["software-dev", "customer-service", "design", "marketing", "sales", "product",
                       "business", "data", "devops", "finance-legal", "hr", "qa", "writing", "all-others"]
RAW_PATH.parent.mkdir(parents=True, exist_ok=True)

## Helpers
`get_json` retries on failure. The response is decoded explicitly as UTF-8 to avoid broken characters such as `MecÃ¡nico`.

In [9]:
def get_json(url, params=None, retries=3, pause=5):
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, params=params, headers=HEADERS, timeout=30)
            response.raise_for_status()
            return json.loads(response.content.decode("utf-8"))
        except (requests.RequestException, ValueError) as error:
            print(f"  attempt {attempt}/{retries} failed: {error}")
            time.sleep(pause * attempt)
    return None


def iso_utc(text):
    """Make every date look the same: 2026-10-05T10:00:00+00:00 (Remotive dates have no timezone)."""
    if not text:
        return np.nan
    text = str(text)
    return text if re.search(r"(Z|[+-]\d\d:?\d\d)$", text) else text + "+00:00"


def parse_salary(text):
    """'$40,000 - $50,000' or '$60k-80k' -> (40000.0, 50000.0). Unclear values -> NaN."""
    if not text:
        return np.nan, np.nan
    cleaned = str(text).lower().replace(",", "")
    numbers = [float(n) * (1000 if k else 1)
               for n, k in re.findall(r"(\d+(?:\.\d+)?)\s*(k)?", cleaned)]
    if not numbers or max(numbers) < 1000:          # hourly or unclear unit
        return np.nan, np.nan
    return min(numbers), max(numbers)

## Source 1: RemoteOK
The first item of the response is a legal notice (it has no `position`), so it is skipped.

In [10]:
def collect_remoteok():
    data = get_json("https://remoteok.com/api")
    if not data:
        print("RemoteOK: no data")
        return []
    rows = []
    for job in data:
        if "position" not in job:
            continue
        rows.append({
            "id": f"remoteok_{job.get('id')}",
            "date": iso_utc(job.get("date")),
            "company": job.get("company"),
            "position": job.get("position"),
            "tags": job.get("tags") or [],
            "description": job.get("description"),
            "location": job.get("location") or np.nan,
            "salary_min": job.get("salary_min", 0),
            "salary_max": job.get("salary_max", 0),
            "apply_url": job.get("apply_url"),
            "url": job.get("url"),
            "source": "remoteok",
            "category": np.nan,
        })
    print(f"RemoteOK: {len(rows)} jobs")
    return rows

## Source 2: Remotive
First we download Remotive's official category list (printed below). Then one request per category, with a pause between requests.

In [11]:
def get_categories():
    data = get_json("https://remotive.com/api/remote-jobs/categories")
    found = []
    if isinstance(data, dict):
        for value in data.values():
            if isinstance(value, list):
                for item in value:
                    if isinstance(item, dict):
                        slug = item.get("slug") or item.get("name")
                        if slug:
                            found.append(slug)
    if not found:
        print("Could not read the category list: using FALLBACK_CATEGORIES")
        found = list(FALLBACK_CATEGORIES)
    found = list(dict.fromkeys(found))              # remove repeats, keep order
    print(f"{len(found)} categories:", found)
    return found


def collect_remotive(categories):
    rows = []
    for category_slug in categories:
        time.sleep(PAUSE_SECONDS)                   # at most 2 requests per minute
        data = get_json("https://remotive.com/api/remote-jobs",
                        params={"category": category_slug, "limit": LIMIT_PER_CATEGORY})
        if not data:
            print(f"Remotive {category_slug}: no data")
            continue
        jobs = data.get("jobs", [])
        print(f"Remotive {category_slug}: {len(jobs)} jobs")
        for job in jobs:
            low, high = parse_salary(job.get("salary"))
            category = job.get("category") or ""
            tags = list(job.get("tags") or [])
            if category:
                tags.append(category.lower())
            rows.append({
                "id": f"remotive_{job.get('id')}",
                "date": iso_utc(job.get("publication_date")),
                "company": job.get("company_name"),
                "position": job.get("title"),
                "tags": tags,
                "description": job.get("description"),
                "location": job.get("candidate_required_location") or np.nan,
                "salary_min": low,
                "salary_max": high,
                "apply_url": job.get("url"),
                "url": job.get("url"),
                "source": "remotive",
                "category": category or np.nan,
            })
    return rows

## Run the collection

In [12]:
collected_at = datetime.now(timezone.utc).isoformat(timespec="seconds")

new_rows = collect_remoteok() + collect_remotive(get_categories())
new = pd.DataFrame(new_rows)
new["collected_at"] = collected_at
new = new.reindex(columns=COLUMNS)
new = new.drop_duplicates(subset="id").drop_duplicates(subset="url")
print("\nNew jobs from this run (after removing repeats):", len(new))

RemoteOK: 99 jobs
30 categories: ['software-development', 'customer-service', 'design', 'marketing', 'sales', 'product', 'project-management', 'artificial-intelligence', 'data', 'devops', 'finance', 'human-resources', 'qa', 'writing', 'legal', 'medical', 'education', 'account-management', 'business-development', 'communications', 'compliance', 'engineering', 'information-technology', 'knowledge-management', 'operations', 'research', 'strategy', 'supply-chain', 'travel-hospitality', 'all-others']
Remotive software-development: 17 jobs
Remotive customer-service: 17 jobs
Remotive design: 17 jobs
Remotive marketing: 17 jobs
Remotive sales: 17 jobs
Remotive product: 17 jobs
Remotive project-management: 17 jobs
Remotive artificial-intelligence: 17 jobs
Remotive data: 17 jobs
Remotive devops: 17 jobs
Remotive finance: 17 jobs
Remotive human-resources: 17 jobs
Remotive qa: 17 jobs
Remotive writing: 17 jobs
Remotive legal: 17 jobs
Remotive medical: 17 jobs
Remotive education: 17 jobs
Remotive a

## Merge with the existing raw file
The old file (RemoteOK only, without `source`) is upgraded: ids become `remoteok_<id>` so they match the new ids and duplicates are detected.
A backup of the old file is saved first.

In [15]:
if RAW_PATH.exists():
    old = pd.read_csv(RAW_PATH, encoding="utf-8")
    old.to_csv(RAW_PATH.with_name("jobs_raw_backup.csv"), index=False, encoding="utf-8")   # safety copy

    if "source" not in old.columns:
        old["source"] = "remoteok"
    old["id"] = old["id"].astype(str)
    is_plain = ~old["id"].str.contains("_")
    old.loc[is_plain, "id"] = old.loc[is_plain, "source"] + "_" + old.loc[is_plain, "id"]
    old = old.reindex(columns=COLUMNS)
else:
    old = pd.DataFrame(columns=COLUMNS)

merged = pd.concat([old, new], ignore_index=True)
merged = merged.drop_duplicates(subset="id", keep="first")       # keep the oldest copy
merged = merged.drop_duplicates(subset="url", keep="first")

added = len(merged) - len(old)
print(f"before: {len(old)} | added: {added} | total: {len(merged)}")

merged.to_csv(RAW_PATH, index=False, encoding="utf-8")
print("saved ->", RAW_PATH)

before: 117 | added: 0 | total: 117
saved -> ../data/raw/jobs_raw.csv


## Progress check

In [18]:
print(merged["source"].value_counts().to_string())
print()
print("Remotive jobs per category:")
print(merged["category"].value_counts().to_string())
# print(f"\nTotal jobs: {len(merged)} / target {TARGET_JOBS}")
# if len(merged) < TARGET_JOBS:
#     print("Not there yet: run this notebook again on another day.")
# else:
#     print("Target reached: continue with 02_data_cleaning.")
# merged.head()

source
remoteok    99
remotive    18

Remotive jobs per category:
category
Software Development       8
Artificial Intelligence    2
Writing                    1
All others                 1
Design                     1
Customer Service           1
Data and Analytics         1
Marketing                  1
Sales                      1
Information Technology     1
